# Kaggle Music Generator — acestep.cpp CUDA worker
This notebook builds the C++ engine with NVIDIA CUDA, downloads the Q8_0 Turbo model set, then polls GitHub Issues.

Before running, add a Kaggle Notebook Secret named `GITHUB_TOKEN` with access to Issues and Contents for this repository. GPU and Internet must be enabled.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
print('Python:', sys.version)
subprocess.run(['nvidia-smi'], check=True)
if shutil.which('nvcc') is None:
    raise RuntimeError('CUDA compiler nvcc is not available in this Kaggle GPU image.')
print('CUDA compiler:', subprocess.check_output(['nvcc', '--version'], text=True).splitlines()[-1])

In [ ]:
# Install the C++ build toolchain and the Hugging Face download CLI used by models.sh.
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','ninja-build','git','pkg-config','libcurl4-openssl-dev'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','hf','six'], check=True)
subprocess.run(['hf','--help'], check=True, stdout=subprocess.DEVNULL)
print('Build tools and model downloader installed.')

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
os.environ['GITHUB_REPOSITORY'] = 'unisocsu/kaggle-music-generator'
os.environ['ACESTEP_CPP_ROOT'] = '/kaggle/working/acestep.cpp'
os.environ['ACESTEP_MODELS_DIR'] = '/kaggle/working/acestep.cpp/models'
os.environ['ACESTEP_DEVICE'] = 'CUDA0'
print('GitHub token loaded from Kaggle Secrets (value hidden).')

In [ ]:
engine_dir = Path(os.environ['ACESTEP_CPP_ROOT'])
if not engine_dir.exists():
    subprocess.run(['git','clone','--depth','1','--recurse-submodules','https://github.com/ServeurpersoCom/acestep.cpp.git',str(engine_dir)], check=True)
else:
    subprocess.run(['git','submodule','update','--init','--recursive'], cwd=engine_dir, check=True)
print('acestep.cpp source ready:', engine_dir)

In [ ]:
# Build specifically for NVIDIA T4 (compute capability 7.5).
subprocess.run(['cmake','-S','.','-B','build','-DGGML_CUDA=ON','-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_BUILD_TYPE=Release'], cwd=engine_dir, check=True)
subprocess.run(['cmake','--build','build','--config','Release','-j2'], cwd=engine_dir, check=True)
for binary in ('ace-lm','ace-synth'):
    path = engine_dir / 'build' / binary
    if not path.is_file(): raise RuntimeError(f'Missing compiled binary: {path}')
print('CUDA binaries built successfully.')

In [ ]:
# Q8_0 quality-oriented model set: 4B LM + Turbo DiT + text encoder + BF16 VAE (~7.7 GB).
subprocess.run(['bash','-lc','./models.sh --lm 4B --quant Q8_0'], cwd=engine_dir, check=True)
print('Downloaded GGUF models:', [p.name for p in (engine_dir/'models').glob('*.gguf')])

In [ ]:
# Start the long-running Issues worker. Models are cached on disk for this Kaggle session.
repo_dir = Path('/kaggle/working/kaggle-music-generator')
if not repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/unisocsu/kaggle-music-generator.git',str(repo_dir)], check=True)
worker = repo_dir / 'kaggle' / 'worker.py'
subprocess.run([sys.executable,str(worker)], cwd=engine_dir, env=os.environ.copy(), check=True)